# Projeto Fundamentos de Aprendizagem Automática

**Grupo 22**

### Membros

- **jiyi Li** (62244)
    
- **oujie Wu** (62228)
    
- **josé lourenço** (62817)
    

  

## Problema

**Estimar a nota final** com base em variáveis comportamentais:

- `student_hour`
    
- `gaming_hours`
    
- `sleep_hours`
    
- `exercise_minutes`
    
- ...
    

        

## Variáveis-Alvo (Targets)

### 1. Cenário de Regressão (y)

Estimar **Exam Score** (nota de exame). Por exemplo, um valor contínuo entre **0 e 100**.

### 2. Cenário de Classificação (y)

Transformar a variável _Exam Score_ em categorias discretas e interpretáveis. Com base na distribuição dos dados ou em critérios pedagógicos, podemos definir:

**Classificação multiclass (Nota entre 0 até 65):**

|**Categoria**|**Intervalo (Exam Score)**|
|---|---|
|**Insuficiente**|0 a 24|
|**Suficiente**|25 a 39|
|**Bom**|40 a 65|



## Stakeholders

- Os próprios estudantes
    
- A instituição de ensino / Escola
    
- Conselheiros / Tutores
    
- Serviços de Apoio ao Estudante







### Ler ficheiro obter feature Variables e Target Variables

In [ ]:
import pandas as pd
import numpy as np
from sklearn import tree
from sklearn.model_selection import KFold, train_test_split
import io
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pydot
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree



# ler ficherio de stufent_records
df_student_records = pd.read_csv('student_records_full.csv')
df_colunas = df_student_records.columns





df_y = df_student_records['exam_score']
df_y_numerico = df_y.copy()  

# print("df_y_numerico:")
# print(df_y_numerico)




def categorizar_exame_score(score):
    if score < 25:
        return 'Insuficiente'  # 挂科风险组
    elif 25 <= score < 40:
        return 'Suficiente'    # 普通及格组
    else:
        return 'Bom' # 优秀组 (合并原本的 Bom 和 Muito Bom)

df_y_categorico = df_y.apply(categorizar_exame_score)




# OrdinalEncoder (categories=categories) para transformar as categorias em números ordinais
categories = [["Insuficiente", "Suficiente", "Bom"]]
oe = OrdinalEncoder(categories=categories)
df_y_categorico_ordinais = oe.fit_transform(df_y_categorico.to_frame())


# print("df_y_categorico:")
# print(df_y_categorico)

# print("df_y_categorico_ordinais:")
# print(df_y_categorico_ordinais)

print(df_y_categorico.value_counts())



### Cenário de Classificação para Árvore de Decisão

In [ ]:
df_student_records.drop(columns=["student_id", "exam_score","productivity_score","burnout_level","focus_index"], inplace=True)
# print(df_student_records.columns)
df_categorico = df_student_records[["gender","academic_level","internet_quality"]]
df_student_records_copy = df_student_records.copy()
# df_student_records.drop(columns=["gender", "academic_level", "internet_quality"], inplace=True)


#(One-Hot Encoding)
df_X = pd.get_dummies(df_student_records_copy, columns=df_categorico.columns.values)
# print(df_X.columns)



X = df_X.values
y = df_y_categorico_ordinais


# Utilizando three/way split
X_learning, X_test, y_learning, y_test = train_test_split(
    X, y, test_size=0.2, random_state=7,stratify=y)

X_train, X_val, y_train, y_val = train_test_split(
    X_learning, y_learning, test_size=0.25, random_state=7,stratify=y_learning)

# print("Training set:", X_train.shape)
# print("Validation set:", X_val.shape)
# print("Test set:", X_test.shape)

# Treino e Avaliação de Modelos Base
param_grid = {
"criterion": ["gini", "entropy"],
"max_depth": [1, 2, 3, 4, 5, 6, None],
"min_samples_split": [2, 5, 10, 20],
"min_samples_leaf": [1, 2, 5, 10],
}

grid = GridSearchCV(
estimator=tree.DecisionTreeClassifier(random_state=7),#固定随机种子确保可重复
param_grid=param_grid,#上面定义的超参数网格。
scoring="accuracy",#用准确率作为模型评估指标
cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=7),#表示使用分层 5 折交叉验证，每折保持原始数据的类别比例，且打乱数据后划分。
n_jobs=-1#使用所有可用的 CPU 核心并行计算，加速搜索过程。
)

grid.fit(X_learning, y_learning) # 仅在学习集上训练和验证

print("Best parameters:", grid.best_params_)#返回交叉验证中平均得分最高的参数组合。
print("Best mean CV accuracy:", round(grid.best_score_, 3))#返回最佳参数组合对应的


#utilizando bestest parameters para treinar o modelo final
final_model = grid.best_estimator_ #- `grid.best_estimator_` 是 `GridSearchCV` 在找到最佳超参数后，使用**整个学习集**（`X_learning, y_learning`）重新训练得到的模型。
y_test_pred = final_model.predict(X_test)
y_test_prob = final_model.predict_proba(X_test)[:, 1]
final_metrics = pd.Series({
    "Accuracy": accuracy_score(y_test, y_test_pred),
    "Precision (macro)": precision_score(y_test, y_test_pred, average='macro'),
    "Precision (weighted)": precision_score(y_test, y_test_pred, average='weighted'),
    "Recall (macro)": recall_score(y_test, y_test_pred, average='macro'),
    "Recall (weighted)": recall_score(y_test, y_test_pred, average='weighted'),
    "F1-score (macro)": f1_score(y_test, y_test_pred, average='macro'),
    "F1-score (weighted)": f1_score(y_test, y_test_pred, average='weighted'),
})

final_metrics.round(3)






In [ ]:
plt.figure(figsize=(50, 20))
plot_tree(
final_model,
feature_names=df_X.columns,
class_names=df_colunas,
filled=True,
rounded=True,
fontsize=8,
)

plt.title("Final selected decision tree")
plt.show()


### Cenário de Regressão para Árvore de Decisão

In [ ]:
import pandas as pd
import numpy as np
from sklearn import tree
from sklearn.model_selection import KFold, train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y = df_y_numerico
X = df_X.values
X_learning, X_test, y_learning, y_test = train_test_split(
    X, y, test_size=0.2, random_state=7
)

X_train, X_val, y_train, y_val = train_test_split(
    X_learning, y_learning, test_size=0.25, random_state=7
)

param_grid = {
"criterion": ["squared_error", "absolute_error"],
"max_depth": [1, 2, 3, 4, 5, 6, None],
"min_samples_split": [2, 5, 10, 20],
"min_samples_leaf": [1, 2, 5, 10],
}

grid = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=7), # 使用回归树
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error", # 优化目标：最小化 RMSE
    cv=KFold(n_splits=5, shuffle=True, random_state=7), # 使用普通的 KFold
    n_jobs=-1
)

grid.fit(X_learning, y_learning) # 在 learning 集上训练和验证

print("Best parameters:", grid.best_params_)
# sklearn 的 gridsearch 会把误差变成负数，所以提取时要加负号取反
print("Best mean CV RMSE:", round(-grid.best_score_, 3))

# 7. 在测试集上做最终评估 (Avaliação no Conjunto de Teste)
final_model = grid.best_estimator_
y_test_pred = final_model.predict(X_test)

# 计算回归专属的评估指标
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae = mean_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)

print("\n--- Métricas no Conjunto de Teste (Test Set Metrics) ---")
print(f"RMSE (Root Mean Squared Error): {round(rmse, 3)}")
print(f"MAE (Mean Absolute Error): {round(mae, 3)}")
print(f"R2 Score: {round(r2, 3)}")



In [ ]:
plt.figure(figsize=(50, 20))
plot_tree(
final_model,
feature_names=df_X.columns,
class_names=df_colunas,
filled=True,
rounded=True,
fontsize=8,
)

plt.title("Final selected decision tree")
plt.show()

### Regresao linear

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score, max_error
from scipy.stats import pearsonr

y = df_y_numerico
df_student_records_numerico = df_student_records.copy()
# print(df_student_records.columns)
df_student_records_numerico.drop(columns=["gender", "academic_level", "internet_quality"], inplace=True)
# print(df_student_records_numerico.columns)




df_xx = pd.get_dummies(df_student_records_copy, columns=df_categorico.columns.values, drop_first=True)
# print(df_X.columns)
# print(df_xx.columns)

X_train, X_test, y_train, y_test = train_test_split(
    df_X, y, test_size=0.2, random_state=7
)

# Vamos usar StandardScaler com cordo formulario: z = (x - u) / s
scaler = StandardScaler()

# 为了不破坏原来的表，我们给训练集和测试集各做一个副本
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# 魔法就在这里：我们只把 cols_numericas 抽出来缩放，然后再塞回原来的表里！
# 注意：X_train 用 fit_transform，X_test 只能用 transform！
X_train_scaled[df_student_records_numerico.columns] = scaler.fit_transform(X_train[df_student_records_numerico.columns])
X_test_scaled[df_student_records_numerico.columns] = scaler.transform(X_test[df_student_records_numerico.columns])




reg = LinearRegression().fit(X_train_scaled, y_train)
print("The score is: ", reg.score(X_train_scaled, y_train))
print("The alpha is: ", reg.intercept_)
print("The other parameters are: ")
for i, beta in enumerate(reg.coef_):
    print("\t B%d -> %9.3f"% (i+1, beta))





def printRegStatistics(truth, preds):
    print("The rmse is: ", root_mean_squared_error(truth, preds)) #值越小，模型性能越好
    print("The Mean Absolute Error is: ", mean_absolute_error(truth, preds))#值越小，模型性能越好
    print("The Coefficient of Determination is: ", r2_score(truth, preds))#值越大（接近1），模型性能越好
    corr, pval = pearsonr(truth, preds)
    print("The Pearson Correlation Score is: %6.4f (p-value=%e)\n"%(corr,pval))
    print("The Maximum Error is: ", max_error(truth, preds))


preds=reg.predict(X_test_scaled)
printRegStatistics(y_test, preds)
plt.figure(figsize=(7,7))
plt.scatter(preds, y_test)
plt.grid()
# 将完美预测的红线改回正常的 0-100 分区间
plt.plot([0, 100], [0, 100], c="r", linestyle='--', linewidth=2)

# 强制限制 X 轴和 Y 轴的显示范围，切掉多余的空白
plt.xlim(0, 100)
plt.ylim(0, 100)
plt.xlabel("predictions")
plt.ylabel("real values")
plt.show()









